# 01 - EDA ve On Isleme

Bu notebook, kalp yetmezligi veri setinin ilk incelemesini ve tum modellerde yeniden kullanilan ortak veri hazirlama adimlarini icerir.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_utils import RANDOM_SEED, prepare_datasets, save_preprocessing_artifacts, summarize_dataframe
from src.visualization import plot_class_distribution, plot_correlation_heatmap


In [ ]:
DATA_PATH = PROJECT_ROOT / 'data' / 'heart_failure_clinical_records_dataset.csv'
OUTPUT_FIGURES = PROJECT_ROOT / 'outputs' / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'

df = pd.read_csv(DATA_PATH)
summary = summarize_dataframe(df)
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
print('Target class distribution:')
print(df['DEATH_EVENT'].value_counts())
df.describe().T

In [ ]:
correlation_matrix = df.corr(numeric_only=True)
correlation_matrix['DEATH_EVENT'].sort_values(ascending=False)

In [ ]:
numeric_columns = ['age', 'creatinine_phosphokinase', 'ejection_fraction', 'platelets', 'serum_creatinine', 'serum_sodium', 'time']
outlier_rows = []
for column in numeric_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = int(((df[column] < lower) | (df[column] > upper)).sum())
    outlier_rows.append({'feature': column, 'outlier_count': outlier_count, 'lower_bound': lower, 'upper_bound': upper})

pd.DataFrame(outlier_rows)

Kisa yorum: Veri setinde eksik ve duplicate gozlem yoktur. Hedef sinif tam dengeli degildir ancak ileri derecede bozuk da degildir. `creatinine_phosphokinase`, `platelets` ve `serum_creatinine` degiskenlerinde aykiri degerler vardir; bunlar klinik anlam tasiyabilecegi icin silinmemis, bunun yerine tum modellerde standardizasyon uygulanmistir.

In [ ]:
plot_class_distribution(df['DEATH_EVENT'], OUTPUT_FIGURES / 'target_class_distribution.png', 'Heart Failure Target Distribution')
plot_correlation_heatmap(correlation_matrix, OUTPUT_FIGURES / 'correlation_heatmap.png', 'Heart Failure Feature Correlation Heatmap')

In [ ]:
split = prepare_datasets(DATA_PATH, random_state=RANDOM_SEED)
save_preprocessing_artifacts(split, MODELS_DIR)

print('Train shape:', split.X_train.shape)
print('Validation shape:', split.X_val.shape)
print('Test shape:', split.X_test.shape)
print('Train class distribution:', pd.Series(split.y_train).value_counts().to_dict())
print('Validation class distribution:', pd.Series(split.y_val).value_counts().to_dict())
print('Test class distribution:', pd.Series(split.y_test).value_counts().to_dict())

In [ ]:
scaled_train_df = pd.DataFrame(split.X_train, columns=split.feature_names)
scaled_train_df.head()